# Notebook 4: Agent Lifecycle, Identity, Tools & Production Patterns
## From Development to Production — The Complete Agent Engineering Guide

**Sources:**
- [Agent Development Lifecycle](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/development-lifecycle)
- [Agent Identity Concepts](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/agent-identity)
- [Hosted Agents](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents)
- [Build a Workflow](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/workflow)
- [Agent Applications](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/agent-applications)
- [Retrieval Augmented Generation (RAG)](https://learn.microsoft.com/en-us/azure/foundry/concepts/retrieval-augmented-generation)
- [Vector Stores](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/vector-stores)
- [Runtime Components](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/runtime-components)
- [Configure Agent](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/configure-agent)
- [Tool Catalog](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/tool-catalog)
- [Tool Best Practices](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/tool-best-practice)
- [Private Tool Catalog](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/private-tool-catalog)
- [Toolbox](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/toolbox)
- [Structured Inputs](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/structured-inputs)
- [Azure AI Search Tool](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/ai-search)
- [Code Interpreter](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/code-interpreter)
- [Function Calling](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling)

---

This notebook covers:
1. Agent Development Lifecycle (8-step checklist)
2. Agent Types: Prompt-based, Workflow, Hosted
3. Runtime Components: Agents, Conversations, Responses
4. Agent Identity & Authentication (Entra ID)
5. Agent Applications — Publishing & Endpoints
6. Configure Agent — Version Routing, Protocols, Auth Schemes
7. RAG Concepts & Vector Stores
8. Tool Catalog — Built-in & Custom Tools
9. Tool Best Practices & tool_choice
10. Private Tool Catalog (Azure API Center)
11. Toolbox — Centralized Tool Management
12. Structured Inputs — Runtime Customization
13. Azure AI Search Tool (with code)
14. Code Interpreter (with code)
15. Function Calling (with code)
16. Interview Q&A

## Master Architecture Diagram

```
┌──────────────────────────────────────────────────────────────────────────────────────┐
│                     MICROSOFT FOUNDRY AGENT SERVICE — FULL PICTURE                   │
│                                                                                      │
│  ┌──────────────────────── DEVELOPMENT ─────────────────────────────┐                 │
│  │                                                                  │                 │
│  │  1. Choose Agent Type    2. Create & Test    3. Add Tools & Data │                 │
│  │  ┌──────────┐           ┌───────────┐       ┌─────────────────┐ │                 │
│  │  │ Prompt   │           │ Playground│       │ Code Interpreter│ │                 │
│  │  │ Workflow │           │ SDK / CLI │       │ File Search     │ │                 │
│  │  │ Hosted   │           │ REST API  │       │ AI Search       │ │                 │
│  │  └──────────┘           └───────────┘       │ Web Search      │ │                 │
│  │                                             │ Function Calling│ │                 │
│  │  4. Version   5. Trace   6. Evaluate        │ MCP / OpenAPI   │ │                 │
│  │  ┌────────┐  ┌───────┐  ┌──────────┐       │ A2A / Toolbox   │ │                 │
│  │  │Immut.  │  │Latency│  │Quality & │       └─────────────────┘ │                 │
│  │  │Versions│  │Tools  │  │Safety    │                           │                 │
│  │  └────────┘  └───────┘  └──────────┘                           │                 │
│  └──────────────────────────────────────────────────────────────────┘                 │
│                                    │                                                  │
│                                    ▼                                                  │
│  ┌──────────────────────── PRODUCTION ──────────────────────────────┐                 │
│  │                                                                  │                 │
│  │  7. Publish (Agent Application)     8. Monitor & Iterate         │                 │
│  │  ┌─────────────────────────────┐   ┌──────────────────────────┐  │                 │
│  │  │ Stable Endpoint             │   │ App Insights / Traces    │  │                 │
│  │  │ Distinct Entra Agent ID     │   │ Quality Dashboards       │  │                 │
│  │  │ RBAC on Application         │   │ Version Updates          │  │                 │
│  │  │ Teams / M365 / Custom Apps  │   │ Rollback Capability      │  │                 │
│  │  └─────────────────────────────┘   └──────────────────────────┘  │                 │
│  └──────────────────────────────────────────────────────────────────┘                 │
│                                                                                      │
│  ┌─────────── IDENTITY ──────────┐   ┌─────────── TOOLS ───────────┐                 │
│  │ Agent Identity Blueprint      │   │ Toolbox (MCP endpoint)      │                 │
│  │ ├── Shared (dev/project)      │   │ Private Tool Catalog        │                 │
│  │ └── Distinct (published)      │   │ Structured Inputs           │                 │
│  │ OAuth 2.0 Token Exchange      │   │ tool_choice (auto/required) │                 │
│  │ Federated Credentials         │   │ Vector Stores (File Search) │                 │
│  └───────────────────────────────┘   └─────────────────────────────┘                 │
└──────────────────────────────────────────────────────────────────────────────────────┘
```

---
## Prerequisites

| Requirement | Description | Status |
|---|---|---|
| **Azure Subscription** | Active subscription | [ ] |
| **Foundry Project** | Created in Notebook 03 | [ ] |
| **Python 3.10+** | Runtime | [ ] |
| **Packages** | `azure-ai-projects>=2.0.0`, `azure-identity`, `openai` | [ ] |
| **Model Deployed** | gpt-4.1-mini or similar | [ ] |
| **Azure CLI** | `az login` completed | [ ] |

In [ ]:
# Install required packages
!pip install azure-ai-projects>=2.0.0 azure-identity openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT", "your_project_endpoint")

if PROJECT_ENDPOINT == "your_project_endpoint":
    print("WARNING: Set PROJECT_ENDPOINT in your .env file.")
    print("Format: https://<resource_name>.ai.azure.com/api/projects/<project_name>")
else:
    print(f"Project endpoint: {PROJECT_ENDPOINT[:60]}...")

---
# PART 1: AGENT DEVELOPMENT LIFECYCLE
---

## 1.1 The 8-Step Lifecycle Checklist

The agent development lifecycle spans from initial creation through production monitoring.

| Step | Action | Details |
|------|--------|--------|
| **1** | Choose an agent type | Prompt-based, Workflow, or Hosted |
| **2** | Create & start testing | Iterate in playground or code |
| **3** | Add tools & data | Attach retrieval tools, APIs; validate before saving |
| **4** | Save as versions | Immutable snapshots; compare across versions |
| **5** | Debug with tracing | Confirm tool calls, inspect latency, validate behavior |
| **6** | Evaluate quality & safety | Run repeatable evaluations to catch regressions |
| **7** | Publish & integrate | Create Agent Application with stable endpoint |
| **8** | Monitor & iterate | Track performance in production, update & republish |

### Common Pitfalls

| Pitfall | Why It Matters |
|---------|---------------|
| **Unsaved changes are temporary** | You lose them if you leave the portal; save to version for history |
| **Tools must be configured before saving** | Auth/connection must be complete before saving agent version |
| **Publishing requires permission updates** | Published agent gets a NEW identity; must reassign RBAC roles |

## 1.2 Three Agent Types in Microsoft Foundry

```
┌────────────────────────────────────────────────────────────────────────┐
│                        AGENT TYPES                                     │
├──────────────────┬─────────────────────┬───────────────────────────────┤
│   PROMPT-BASED   │     WORKFLOW        │         HOSTED (Preview)      │
├──────────────────┼─────────────────────┼───────────────────────────────┤
│ Declarative      │ Visual orchestrator │ Containerized custom code     │
│ Single agent     │ Multi-agent         │ Any framework                 │
│ Model+instructions│ Branching logic    │ Custom protocols              │
│ +tools+prompts   │ Human-in-loop      │ Own compute resources         │
│ Portal editable  │ Power Fx formulas  │ VM-isolated sandboxes         │
│ Versioned        │ YAML export        │ Scale-to-zero w/ state resume │
└──────────────────┴─────────────────────┴───────────────────────────────┘
```

### When to Use Each Type

| Scenario | Best Type | Reason |
|----------|-----------|--------|
| Simple Q&A with tools | **Prompt-based** | Fastest to build, portal-editable |
| Multi-step approval workflows | **Workflow** | Visual builder, if/else logic, human-in-loop |
| Custom framework (LangGraph, Semantic Kernel) | **Hosted** | Bring your own code, any framework |
| Webhook receiver (GitHub, Stripe) | **Hosted** | Invocations protocol for arbitrary JSON payloads |
| Multi-agent coordination | **Workflow** or **Hosted** | Workflow for low-code; Hosted for pro-code |

## 1.3 Version Management

Every change to an agent creates a new **immutable version**. Key capabilities:

| Feature | Description |
|---------|-------------|
| **Immutability** | Each saved version is frozen; modifications require a new version |
| **Draft State** | Test unsaved changes in playground; lose them if you leave |
| **Version Targeting** | Direct requests to specific versions for controlled rollback |
| **Comparison** | Compare agent setup, chat output, or YAML between any two versions |

> **Interview Tip:** Agent names are immutable after creation. In code, reference agents as `<agent_name>:<version>`.

---
# PART 2: RUNTIME COMPONENTS — AGENTS, CONVERSATIONS, RESPONSES
---

## 2.1 The Three Core Runtime Components

```
┌─────────────────────────────────────────────────────────────────┐
│                    RUNTIME COMPONENT MODEL                      │
│                                                                 │
│  ┌──────────┐     ┌──────────────┐     ┌────────────────────┐  │
│  │  AGENT   │────▶│ CONVERSATION │────▶│    RESPONSE        │  │
│  │          │     │              │     │                    │  │
│  │ Model    │     │ History      │     │ Output items       │  │
│  │ Instruct.│     │ Multi-turn   │     │ Tool calls         │  │
│  │ Tools    │     │ Context      │     │ Streaming support  │  │
│  │ Version  │     │ (Optional)   │     │ Citations          │  │
│  └──────────┘     └──────────────┘     └────────────────────┘  │
│                                                                 │
│  Agent = persisted definition (model + instructions + tools)    │
│  Conversation = persisted history across turns (optional)       │
│  Response = output the agent produces for given input           │
└─────────────────────────────────────────────────────────────────┘
```

### How They Work Together

1. **Create an agent** — define model, instructions, tools
2. **Create a conversation** (optional) — maintains history across turns
3. **Generate a response** — agent processes input + history, may call tools
4. **Check response status** — monitor until finished (streaming/background)
5. **Retrieve the response** — display output to user

> **Key Change:** Agents are now identified by `name` + `version` (no more GUID-based `AgentID`).

In [ ]:
# 2.2 Create an Agent with a Tool
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, WebSearchTool

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
openai_client = project.get_openai_client()

# Create agent with web search tool
agent = project.agents.create_version(
    agent_name="lifecycle-demo-agent",
    definition=PromptAgentDefinition(
        model="gpt-4.1-mini",
        instructions="You are a helpful assistant that can search the web for current information.",
        tools=[WebSearchTool()],
    ),
)
print(f"Agent created: {agent.name}, Version: {agent.version}")

In [ ]:
# 2.3 Multi-Turn Conversation with an Agent
# Create a conversation to maintain history across turns
conversation = openai_client.conversations.create()
print(f"Conversation: {conversation.id}\n")

# Turn 1
response1 = openai_client.responses.create(
    conversation=conversation.id,
    input="What are the latest features in Microsoft Foundry?",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)
print(f"Turn 1: {response1.output_text[:200]}...\n")

# Turn 2 — Follow-up (agent remembers context)
response2 = openai_client.responses.create(
    conversation=conversation.id,
    input="Which of those features support Python?",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)
print(f"Turn 2: {response2.output_text[:200]}...")

In [ ]:
# 2.4 Streaming Response
stream_response = openai_client.responses.create(
    stream=True,
    input="What is RAG in 2 sentences?",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

print("Streaming: ", end="")
for event in stream_response:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")
print()  # newline at the end

---
# PART 3: AGENT IDENTITY & AUTHENTICATION
---

## 3.1 What Is an Agent Identity?

An **agent identity** is a specialized service principal in **Microsoft Entra ID** designed specifically for AI agents. It provides:

- **Standardized governance** — inventory, audit, and manage agents like workforce identities
- **Tool authentication** — agents authenticate to downstream services (Storage, Cosmos DB, etc.) without embedded secrets
- **Distinction from human identities** — separate tracking from workforce/customer/workload operations

### Key Terms

| Term | Meaning |
|------|--------|
| **Agent Identity** | A special Entra ID service principal representing the agent at runtime |
| **Agent Identity Blueprint** | An Entra ID object governing a *class* of agent identities (template) |
| **`agentIdentityId`** | The ID used when assigning RBAC permissions to the agent |
| **Audience** | OAuth resource identifier for downstream service (e.g., `https://storage.azure.com`) |

### Common Audience Values

| Downstream Service | Audience Value |
|-------------------|---------------|
| Azure Storage | `https://storage.azure.com` |
| Azure Logic Apps | `https://logic.azure.com` |
| Azure Cosmos DB | `https://cosmos.azure.com` |
| Microsoft Graph | `https://graph.microsoft.com` |
| Azure Key Vault | `https://vault.azure.net` |

## 3.2 Runtime Token Exchange (4-Stage OAuth 2.0)

When an agent invokes a tool, this happens automatically:

```
┌─────────────────────────────────────────────────────────────────────┐
│              RUNTIME TOKEN EXCHANGE (Automatic)                     │
│                                                                     │
│  Stage 1: Blueprint Authentication                                  │
│  Agent Service ──(blueprint creds)──▶ Entra ID                      │
│  "I'm authorized to act on behalf of this agent class"              │
│                           │                                         │
│  Stage 2: Agent Identity Token                                      │
│  Entra ID ──(agent identity token)──▶ Agent Service                 │
│  Token identifies the agent as an independent actor                 │
│                           │                                         │
│  Stage 3: Scoped Token Request                                      │
│  Agent Service ──(agent token + audience)──▶ Entra ID               │
│  Requests access token for downstream service                       │
│  (e.g., audience = https://storage.azure.com)                       │
│                           │                                         │
│  Stage 4: Authenticated Tool Call                                   │
│  Agent Service ──(scoped token)──▶ MCP/A2A/Downstream               │
│  Resource validates token + checks RBAC                             │
└─────────────────────────────────────────────────────────────────────┘
```

> **Critical:** An incorrect `audience` value causes auth failures even with correct RBAC. The audience must match the downstream service's resource identifier.

## 3.3 Shared vs. Distinct Identity

```
┌──────────────────────────────────────────────────────────────┐
│                    IDENTITY LIFECYCLE                         │
│                                                              │
│  DEVELOPMENT PHASE              PRODUCTION PHASE             │
│  ┌─────────────────┐            ┌─────────────────────────┐  │
│  │ SHARED IDENTITY │   Publish  │ DISTINCT IDENTITY       │  │
│  │                 │  ────────▶ │                         │  │
│  │ All unpublished │            │ Per agent application   │  │
│  │ agents in project│           │ Own blueprint + identity│  │
│  │ share ONE identity│          │ Independent RBAC        │  │
│  │                 │            │ Separate audit trail    │  │
│  └─────────────────┘            └─────────────────────────┘  │
│                                                              │
│  ⚠️  Permissions do NOT transfer on publish!                 │
│      Must reassign RBAC to the new distinct identity         │
└──────────────────────────────────────────────────────────────┘
```

### Two Authentication Flows

| Flow | Name | How It Works |
|------|------|-------------|
| **Attended** | Delegated / OBO | Agent acts on behalf of a human user (user's permissions) |
| **Unattended** | Application-only | Agent acts under its own authority (own RBAC roles) |

### Blueprint Credential Types

| Type | Trade-offs |
|------|----------|
| **Client secret** | Simplest; requires manual rotation |
| **Certificate** | Stronger; requires cert lifecycle management |
| **Federated credential** | **Recommended for production**; Azure manages rotation automatically |

In [ ]:
# 3.4 Assign RBAC to Agent Identity (CLI example)
# After publishing, you get a new agentIdentityId from the Azure Portal JSON view

print("""
# Assign Storage Blob Data Contributor to agent identity:
az role assignment create \\
    --assignee "<agentIdentityId>" \\
    --role "Storage Blob Data Contributor" \\
    --scope "/subscriptions/<sub-id>/resourceGroups/<rg>/providers/Microsoft.Storage/storageAccounts/<account>"

# Verify the assignment:
az role assignment list \\
    --assignee "<agentIdentityId>" \\
    --scope "<same-scope>" \\
    --output table

# Common role assignments for agent tools:
# ┌───────────────────────────────┬─────────────────────────────────┬──────────────────┐
# │ Tool Scenario                 │ Required Role                   │ Target Scope     │
# ├───────────────────────────────┼─────────────────────────────────┼──────────────────┤
# │ MCP reads/writes blobs        │ Storage Blob Data Contributor   │ Storage account  │
# │ MCP triggers logic apps       │ Logic Apps Standard Operator    │ Logic App        │
# │ A2A queries Cosmos DB         │ Cosmos DB Built-in Data Reader  │ Cosmos DB acct   │
# └───────────────────────────────┴─────────────────────────────────┴──────────────────┘
""")

---
# PART 4: AGENT APPLICATIONS — PUBLISHING TO PRODUCTION
---

## 4.1 What Is Publishing?

Publishing moves an agent from a development asset inside your project to a **managed Azure resource** with:

| Capability | Description |
|-----------|------------|
| **Stable Endpoint** | URL stays the same across version updates |
| **Distinct Identity** | Own Entra agent identity + blueprint |
| **External Sharing** | Grant access without project access |
| **Independent RBAC** | Separate Azure resource with own RBAC scope |
| **Azure Policy** | Governable as ARM resource |
| **M365/Teams Integration** | Distribute to Teams, Microsoft 365 Copilot |

### Agent Application Object Model

```
┌─────────────────────────────────────────────────────────────┐
│  Foundry Project                                            │
│  ├── Agent "sales-bot"                                      │
│  │   ├── Version 1 (immutable snapshot)                     │
│  │   ├── Version 2 (immutable snapshot)                     │
│  │   └── Version 3 (latest)                                 │
│  │                                                          │
│  └── Agent Application (Azure Resource)                     │
│      ├── Stable endpoint URL                                │
│      ├── Distinct Entra agent identity                      │
│      ├── Authorization policy (RBAC / Bot Service)          │
│      ├── Traffic routing policy                             │
│      └── Deployment                                         │
│          ├── References agent version 3                     │
│          ├── Protocol: Responses / Activity                 │
│          └── State: Starting|Running|Stopping|Failed        │
└─────────────────────────────────────────────────────────────┘
```

## 4.2 Configure Agent — New Publishing Model

Every agent now has a **stable endpoint from creation** — no separate publish step required.

### Traffic Routing Policies

| Policy | Behavior |
|--------|----------|
| **Always use latest** (default) | 100% traffic routes to most recently created version |
| **Pinned to specific version** | 100% traffic routes to version you select; new versions don't change what's served |

### Available Protocols

| Protocol | Endpoint Pattern | Best For |
|----------|-----------------|----------|
| **Responses** | `{account}/agents/{agent}/endpoint/protocols/openai/v1/responses` | Most agents — platform manages conversation history |
| **Activity** | `{account}/agents/{agent}/endpoint/protocols/activityprotocol` | Teams / M365 channel delivery |
| **Invocations** | `{account}/agents/{agent}/endpoint/protocols/invocations` | Webhooks, custom payloads, arbitrary JSON |
| **A2A (preview)** | `{account}/agents/{agent}/endpoint/protocols/a2a` | Agent-to-agent delegation |

### Authorization Schemes

| Scheme | Description |
|--------|------------|
| **Entra** | Microsoft Entra ID auth; caller needs Foundry User role |
| **BotService** | Azure Bot Service channel auth (Teams/M365) |
| **BotServiceRbac** | Bot Service + additional RBAC enforcement |

> **Note:** API key authentication is NOT supported for agent endpoints.

In [ ]:
# 4.3 Pin Traffic to a Specific Version (Python SDK)
print("""
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AgentEndpoint,
    FixedRatioVersionSelectionRule,
    VersionSelector,
)
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

with project_client:
    endpoint_config = AgentEndpoint(
        version_selector=VersionSelector(
            version_selection_rules=[
                FixedRatioVersionSelectionRule(agent_version="2", traffic_percentage=100),
            ]
        ),
    )
    
    patched_agent = project_client.beta.agents.patch_agent_details(
        agent_name="my-agent",
        agent_endpoint=endpoint_config,
    )
    print(f"Traffic pinned to version 2 for agent: {patched_agent.name}")
""")

In [ ]:
# 4.4 Invoke a Published Agent Application (Python)
print("""
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

BASE_URL = "https://<foundry-resource>.services.ai.azure.com/api/projects/<project>/applications/<app>/protocols/openai"

openai = OpenAI(
    api_key=get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default"),
    base_url=BASE_URL,
    default_query={"api-version": "2025-11-15-preview"}
)

response = openai.responses.create(input="Write a haiku")
print(response.output_text)

# Note: Only stateless Responses API is supported for published agents.
# The client must store conversation history for multi-turn conversations.
""")

---
# PART 5: HOSTED AGENTS (PREVIEW)
---

## 5.1 What Are Hosted Agents?

Hosted agents are **containerized agentic AI applications** that run on Foundry Agent Service. Unlike prompt-based agents, they are your own code packaged as a container image.

### When to Use Hosted Agents

| Need | Why Hosted |
|------|----------|
| Bring your own code | Use any framework (Agent Framework, LangGraph, Semantic Kernel) |
| Custom protocols | Accept webhooks or non-OpenAI payloads (Invocations protocol) |
| Control compute | Specify CPU/memory (0.25 vCPU/0.5 GiB to 2 vCPU/4 GiB) |
| Stateful workloads | Persist files/state across turns via `$HOME` and `/files` endpoint |

### Two Protocols

| Feature | Responses | Invocations |
|---------|-----------|------------|
| **Best for** | Most agents (chatbot, Q&A, RAG) | Webhooks, custom payloads, async |
| **Payload** | OpenAI-compatible `/responses` | Arbitrary JSON via `/invocations` |
| **Client SDK** | Any OpenAI-compatible SDK | Custom client |
| **Session history** | Platform-managed | You manage (in-memory, Cosmos DB) |
| **Streaming** | Platform-managed events | Raw SSE — you control |
| **Background** | Built-in (`background: true`) | Manual task tracking |

### Session Lifecycle

| State | What Happens |
|-------|-------------|
| **Active** | Compute running; requests routed; `$HOME` and `/files` available |
| **Idle** | No requests for 15 min; compute deprovisioned; state persisted |
| **Resumed** | Same session ID referenced; new compute provisioned; state restored |

- Sessions persist up to **30 days**
- Idle timeout is **15 minutes**
- Per-session **VM-isolated sandboxes**

---
# PART 6: WORKFLOWS — VISUAL MULTI-AGENT ORCHESTRATION
---

## 6.1 What Are Workflows?

Workflows are **UI-based, declarative orchestrators** that coordinate multiple agents and business logic visually.

### Workflow Patterns

| Pattern | Description | Use Case |
|---------|-------------|----------|
| **Human in the loop** | Asks user a question, awaits input | Approvals, clarifying questions |
| **Sequential** | Passes result from one agent to next | Step-by-step pipelines |
| **Group chat** | Dynamic control passing between agents | Escalation, expert handoff |

### Node Types

| Type | Purpose |
|------|--------|
| **Agent** | Invoke a Foundry agent |
| **Logic** | If/else, go to, for each |
| **Data transformation** | Set variable, parse value |
| **Basic chat** | Send message, ask question |

### Power Fx Integration

Workflows use **Power Fx** (Excel-like formulas) for data manipulation:

- Variables use scope prefixes: `System.` (system vars) or `Local.` (local vars)
- Example: `{Upper(Local.UserName)}` converts a user's name to uppercase
- Supports String, Boolean, Number, Record/table, Date/time, Blank types

### Structured JSON Output from Agents

Configure agents in workflows to return structured JSON:

```json
{
  "name": "math_response",
  "schema": {
    "type": "object",
    "properties": {
      "steps": {
        "type": "array",
        "items": {
          "type": "object",
          "properties": {
            "explanation": { "type": "string" },
            "output": { "type": "string" }
          },
          "required": ["explanation", "output"]
        }
      },
      "final_answer": { "type": "string" }
    },
    "required": ["steps", "final_answer"]
  },
  "strict": true
}
```

> **Limitation:** Hosted agents are NOT supported in the workflow designer. Use Microsoft Agent Framework for pro-code orchestration.

---
# PART 7: RAG & VECTOR STORES
---

## 7.1 What Is RAG?

**Retrieval Augmented Generation (RAG)** combines search with LLMs so responses are grounded in your data.

```
┌────────────────────────────────────────────────────────────┐
│                    RAG 3-STEP FLOW                         │
│                                                            │
│  1. RETRIEVE          2. AUGMENT           3. GENERATE     │
│  ┌──────────┐        ┌──────────────┐     ┌────────────┐  │
│  │ User asks │──────▶│ Combine query│────▶│ Model gets  │  │
│  │ question  │       │ + retrieved  │     │ augmented   │  │
│  │           │       │ content into │     │ prompt and  │  │
│  │ Query     │       │ prompt       │     │ generates   │  │
│  │ index/    │       │              │     │ grounded    │  │
│  │ data store│       │ (grounding)  │     │ response    │  │
│  └──────────┘        └──────────────┘     └────────────┘  │
└────────────────────────────────────────────────────────────┘
```

### Key Concepts

| Concept | Definition |
|---------|----------|
| **Grounding data** | Retrieved content provided to the model to reduce hallucination |
| **Index** | Data structure optimized for retrieval (keyword, semantic, vector, hybrid) |
| **Embeddings** | Numeric representations for vector similarity search |
| **System message** | Instructions guiding how the model uses retrieved content |

### Agentic RAG (Modern Approach)

**Agentic retrieval** uses a model to break complex inputs into multiple focused subqueries:

| Feature | Benefit |
|---------|--------|
| **Context-aware query planning** | Follow-up questions retain context |
| **Parallel execution** | Multiple subqueries simultaneously |
| **Structured responses** | Citations, metadata, execution info |
| **Built-in semantic ranking** | Optimal relevance filtering |
| **Optional answer synthesis** | LLM-formulated answers in query response |

### Choose Your Approach

| Approach | When to Use |
|----------|------------|
| **RAG** | Need answers grounded in private/changing data |
| **Fine-tuning** | Need to change model behavior/style, not add knowledge |
| **Agent tools** | Building an agent that needs retrieval as a tool |

## 7.2 Vector Stores

Vector stores give the **file search** tool the ability to search your files. The system automatically:
1. **Chunks** your content into manageable pieces
2. **Converts** each chunk into embeddings
3. **Stores** vectors in an optimized search index
4. **Creates associations** between vectors and original content

### Default Retrieval Settings

| Setting | Default Value |
|---------|-------------|
| Chunk size | 800 tokens |
| Chunk overlap | 400 tokens |
| Embedding model | text-embedding-3-large at 256 dimensions |
| Max chunks in context | 20 |
| Files per vector store | Up to 10,000 |
| Vector stores per agent | 1 max |
| Vector stores per conversation | 1 max |

### Data Location

| Setup Type | Where Files Live |
|-----------|----------------|
| **Basic** | Microsoft-managed storage and search |
| **Standard** | YOUR Azure Blob Storage + Azure AI Search |

### Vector Store Lifecycle

```
Upload Files ──▶ Ingestion (in_progress) ──▶ Ready (completed) ──▶ Expiration
                      │                                               │
                      ├── Always poll before creating responses        │
                      └── 60s max wait fallback for conversation VS    │
                                                                      │
                          Conversation VS: 7-day default expiration ───┘
                          (7 days after last use in response generation)
```

> **Interview Tip:** Always ensure vector store status is `completed` before generating responses. Use SDK polling helpers (`create-and-poll`, `upload-and-poll`).

---
# PART 8: TOOL CATALOG — BUILT-IN & CUSTOM TOOLS
---

## 8.1 What Are Tools?

A **tool** is a capability an agent can invoke during a conversation. The model decides whether to call a tool based on instructions and available tool definitions.

### Built-in Tools

| Tool | Description |
|------|------------|
| **Web Search** | Real-time info from public web with inline citations |
| **Code Interpreter** | Write and run Python in sandboxed environment |
| **File Search** | Augment with knowledge from uploaded files (vector search) |
| **Azure AI Search** | Ground responses with data from existing search index |
| **Azure Functions** | Call your Azure Functions for custom actions |
| **Function Calling** | Define custom functions; your app executes and returns results |
| **Image Generation** | Generate images during conversations (preview) |
| **Browser Automation** | Browser tasks via natural language (preview) |
| **Computer Use** | Interact with computer UIs (preview) |
| **Microsoft Fabric** | Connect to Fabric data agent (preview) |
| **SharePoint** | Chat with private SharePoint documents (preview) |

### Custom Tools

| Tool | Description |
|------|------------|
| **MCP (Model Context Protocol)** | Connect to tools on an MCP server endpoint |
| **OpenAPI** | Connect to external APIs via OpenAPI 3.0/3.1 spec |
| **Agent-to-Agent (A2A)** | Cross-agent communication (preview) |
| **Toolbox** | Bundle multiple tools into single MCP endpoint (preview) |

## 8.2 Tool Best Practices

### Control Tool Calling with `tool_choice`

| Value | Behavior |
|-------|----------|
| `auto` | Model decides whether to call tools |
| `required` | Model MUST call one or more tools |
| `none` | Model does NOT call tools |

### Best Practices Checklist

- [x] Describe what each tool is for in agent instructions
- [x] Add decision rules if tools overlap ("Use File Search before Web Search for internal content")
- [x] Treat tool outputs as **untrusted input** — validate before acting
- [x] Never include keys/tokens/credentials in prompts
- [x] Don't log secrets in traces
- [x] Use `tool_choice="required"` for deterministic tool calling
- [x] Review run traces to confirm tool calls happened

### Troubleshooting Tool Issues

| Issue | Fix |
|-------|-----|
| Agent doesn't call a tool | Confirm tool attached + model supports it + use `tool_choice="required"` |
| Tool returns empty results | Improve descriptions; ensure data is ingested/searchable |
| Tool calls fail | Verify config, auth, endpoint reachability |
| "Tool not supported" error | Tool needs BOTH model AND region support — check both tables |

## 8.3 Private Tool Catalog

Create a **private tool catalog** using **Azure API Center** so only your org's developers can discover and use your MCP server tools.

### Setup Flow

```
1. Create Azure API Center resource
   └── Name becomes catalog name in Foundry Tools
   
2. Register remote MCP servers in API Center
   └── Configure environments and deployments

3. Configure MCP authentication (if needed)
   └── API Key, OAuth, or HTTP bearer token

4. Grant developer access
   └── Assign Azure API Center Data Reader role
   └── Role propagation can take up to 24 hours

5. Verify in Foundry Tools
   └── Build > Tools > Search for your catalog name
```

### Access Model

| Role | Who | Where |
|------|-----|-------|
| Catalog admins | Create/manage catalog | Azure API Center |
| Developers (discover) | View registered MCP servers | Azure API Center RBAC (Data Reader) |
| Developers (use) | Configure and use tools | Foundry project |

## 8.4 Toolbox — Centralized Tool Management (Preview)

A **Toolbox** bundles multiple tools (MCP servers, Web Search, AI Search, Code Interpreter, etc.) into a **single MCP-compatible endpoint**.

### Why Toolbox?

```
WITHOUT TOOLBOX:                          WITH TOOLBOX:
┌──────────┐  ┌──────────┐               ┌──────────┐
│ Agent A  │  │ Agent B  │               │ Agent A  │──┐
│ ├─Tool 1 │  │ ├─Tool 1 │               └──────────┘  │
│ ├─Tool 2 │  │ ├─Tool 2 │               ┌──────────┐  │   ┌──────────────┐
│ └─Tool 3 │  │ └─Tool 3 │               │ Agent B  │──┼──▶│   TOOLBOX    │
└──────────┘  └──────────┘               └──────────┘  │   │ (Single MCP  │
                                          ┌──────────┐  │   │  endpoint)   │
Duplicated config,                        │ Agent C  │──┘   │ Tool 1,2,3   │
duplicated creds,                         └──────────┘      │ Central auth │
no governance                                               └──────────────┘
```

### Key Benefits

| Feature | Benefit |
|---------|--------|
| **Single endpoint** | Any MCP-compatible runtime can consume it |
| **Central auth** | Credential injection, token refresh, policy enforcement |
| **Versioning** | Test new versions, promote to default when ready |
| **No code changes** | Add/remove/reconfigure tools without changing agent code |
| **Framework-agnostic** | Works with Agent Framework, LangGraph, GitHub Copilot SDK, custom code |

In [ ]:
# 8.5 Create a Toolbox (Python SDK)
print("""
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, WebSearchTool

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Create toolbox with web search and MCP tools
toolbox_version = project.beta.toolboxes.create_toolbox_version(
    toolbox_name="my-toolbox",
    description="Toolbox with web search and an MCP server",
    tools=[
        WebSearchTool(),
        MCPTool(
            server_label="myserver",
            server_url="https://your-mcp-server.example.com",
            require_approval="never",
            project_connection_id="my-key-auth-connection",
        ),
    ],
)
print(f"Toolbox: {toolbox_version.name}, version: {toolbox_version.version}")

# Get the MCP endpoint URL for consumption
toolbox_info = project.beta.toolboxes.get_toolbox(toolbox_name="my-toolbox")
mcp_endpoint = toolbox_info.consumer_endpoint
print(f"MCP endpoint: {mcp_endpoint}")
""")

---
# PART 9: STRUCTURED INPUTS — RUNTIME CUSTOMIZATION
---

## 9.1 What Are Structured Inputs?

**Structured inputs** use handlebar template syntax (`{{variableName}}`) to create parameterized agent definitions. Override values at runtime without creating new agent versions.

### Two Categories of Overrides

| Category | What You Can Override |
|----------|--------------------|
| **Instruction overrides** | Agent instructions, response instructions, system messages |
| **Tool resource overrides** | Vector store IDs, file IDs, MCP server URLs, headers, AI Search filters |

### Supported Properties

| Category | Property | Description |
|----------|---------|------------|
| Instructions | Agent `instructions` | Agent-level instruction text |
| Instructions | Response `instructions` | Per-request instructions |
| File Search | `vector_store_ids` | Array of vector store IDs |
| Code Interpreter | `container` / `container.file_ids` | Container or file IDs |
| MCP | `server_label`, `server_url`, `headers` | MCP server configuration |
| Azure AI Search | `filter` | OData filter expression |

In [ ]:
# 9.2 Structured Inputs Example — Personalized Agent Instructions
print("""
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, StructuredInputDefinition
from azure.identity import DefaultAzureCredential

project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
openai = project.get_openai_client()

# Create agent with handlebar templates in instructions
agent = project.agents.create_version(
    agent_name="personalized-agent",
    definition=PromptAgentDefinition(
        model="gpt-4.1-mini",
        instructions=(
            "You are a helpful assistant. "
            "The user's name is {{userName}} and their role is {{userRole}}. "
            "Greet them and confirm their details."
        ),
        structured_inputs={
            "userName": StructuredInputDefinition(
                description="The user's name", required=True, schema={"type": "string"},
            ),
            "userRole": StructuredInputDefinition(
                description="The user's role", required=True, schema={"type": "string"},
            ),
        },
    ),
)

# Pass values at runtime — NO new agent version needed!
conversation = openai.conversations.create()
response = openai.responses.create(
    conversation=conversation.id,
    input="Hello! Can you confirm my details?",
    extra_body={
        "agent_reference": {"name": agent.name, "type": "agent_reference"},
        "structured_inputs": {"userName": "Alice Smith", "userRole": "Senior Developer"},
    },
)
print(response.output_text)
# Output: "Hello Alice Smith! Your name is Alice Smith and your role is Senior Developer."
""")

---
# PART 10: AZURE AI SEARCH TOOL (WITH CODE)
---

## 10.1 How It Works

The Azure AI Search tool connects your agent to an **existing search index**, enabling grounded responses with inline citations.

### Tool Parameters

| Parameter | Required | Notes |
|-----------|---------|-------|
| `project_connection_id` | Yes | Resource ID of project connection to AI Search |
| `index_name` | Yes | Name of the index |
| `top_k` | No | Defaults to 5 |
| `query_type` | No | Default: `vector_semantic_hybrid`. Options: `simple`, `vector`, `semantic`, `vector_simple_hybrid`, `vector_semantic_hybrid` |
| `filter` | No | OData filter applied to all queries |

### Prerequisites

- Azure AI Search index with vector fields + searchable/retrievable text fields
- Project connection to Azure AI Search service
- For keyless auth: **Search Index Data Contributor** + **Search Service Contributor** roles

In [ ]:
# 10.2 Create Agent with Azure AI Search Tool
print("""
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AzureAISearchTool,
    PromptAgentDefinition,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
)

PROJECT_ENDPOINT = "your_project_endpoint"
SEARCH_CONNECTION_NAME = "my-search-connection"
SEARCH_INDEX_NAME = "my-search-index"

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
openai = project.get_openai_client()

# Resolve the connection ID
azs_connection = project.connections.get(SEARCH_CONNECTION_NAME)
connection_id = azs_connection.id

# Create agent with AI Search tool
agent = project.agents.create_version(
    agent_name="search-agent",
    definition=PromptAgentDefinition(
        model="gpt-4.1-mini",
        instructions=\"\"\"You are a helpful assistant. Always provide citations 
        using the tool and render them as: [message_idx:search_idx+source].\"\"\",
        tools=[
            AzureAISearchTool(
                azure_ai_search=AzureAISearchToolResource(
                    indexes=[
                        AISearchIndexResource(
                            project_connection_id=connection_id,
                            index_name=SEARCH_INDEX_NAME,
                            query_type=AzureAISearchQueryType.SIMPLE,
                        ),
                    ]
                )
            )
        ],
    ),
)

# Stream response with citations
stream_response = openai.responses.create(
    stream=True,
    tool_choice="required",
    input="Tell me about mental health services",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

for event in stream_response:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")
    elif event.type == "response.output_item.done":
        if event.item.type == "message":
            for annotation in event.item.content[-1].annotations:
                if annotation.type == "url_citation":
                    print(f"\\nCitation: {annotation.url}")
""")

---
# PART 11: CODE INTERPRETER (WITH CODE)
---

## 11.1 What Is Code Interpreter?

Code Interpreter lets an agent **write and run Python code in a sandboxed environment** for:
- Data analysis and visualization
- Mathematical computation
- Chart/file generation
- Iterative problem-solving

### Key Details

| Feature | Detail |
|---------|--------|
| **Environment** | Microsoft-managed sandbox (Hyper-V isolated) |
| **Session lifetime** | 1 hour active, 30 min idle timeout |
| **Billing** | Per-session charges beyond token-based fees |
| **Isolation** | Each conversation = separate session |
| **Network** | No outbound network requests from sandbox |
| **Packages** | Common data science packages; for custom packages use Custom Code Interpreter |

### Supported File Types

`.c`, `.cpp`, `.csv`, `.docx`, `.html`, `.java`, `.json`, `.md`, `.pdf`, `.php`, `.pptx`, `.py`, `.rb`, `.tex`, `.txt`, `.css`, `.jpeg/.jpg`, `.js`, `.gif`, `.png`, `.tar`, `.ts`, `.xlsx`, `.xml`, `.zip`

In [ ]:
# 11.2 Create Agent with Code Interpreter
print("""
import os
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition, 
    CodeInterpreterTool, 
    AutoCodeInterpreterToolParam,
)

PROJECT_ENDPOINT = "your_project_endpoint"

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
openai = project.get_openai_client()

# Upload a CSV file for analysis
file = openai.files.create(
    purpose="assistants", 
    file=open("quarterly_results.csv", "rb")
)

# Create agent with Code Interpreter
agent = project.agents.create_version(
    agent_name="data-analyst",
    definition=PromptAgentDefinition(
        model="gpt-4.1-mini",
        instructions="You are a data analysis assistant.",
        tools=[CodeInterpreterTool(
            container=AutoCodeInterpreterToolParam(file_ids=[file.id])
        )],
    ),
)

# Request chart generation
conversation = openai.conversations.create()
response = openai.responses.create(
    conversation=conversation.id,
    input="Create a bar chart of operating profit by quarter from the uploaded CSV.",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

# Extract generated file from response annotations
last_message = response.output[-1]
if last_message.type == "message" and last_message.content[-1].annotations:
    citation = last_message.content[-1].annotations[-1]
    if citation.type == "container_file_citation":
        # Download the generated chart
        file_content = openai.containers.files.content.retrieve(
            file_id=citation.file_id, 
            container_id=citation.container_id
        )
        with open(citation.filename, "wb") as f:
            f.write(file_content.read())
        print(f"Chart downloaded: {citation.filename}")
""")

---
# PART 12: FUNCTION CALLING (WITH CODE)
---

## 12.1 How Function Calling Works

Function calling lets you extend agents with **custom capabilities**. The flow:

```
1. DEFINE ──▶ 2. CREATE ──▶ 3. SEND ──▶ 4. EXECUTE ──▶ 5. RESPOND
   Function      Agent        Prompt      Function       Final
   tools         with tools   to agent    locally &      response
   (schema)                               return output
```

### Key Rules

- Runs expire **10 minutes** after creation — submit tool outputs before expiry
- The model writes the function call; **YOUR app executes it**
- Portal shows agents but does NOT execute function calls — use SDK/REST
- Treat function arguments as untrusted; validate before use

In [ ]:
# 12.2 Function Calling — Complete Example
import json
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool
from azure.identity import DefaultAzureCredential

# Define your local function
def get_horoscope(sign: str) -> str:
    """Generate a horoscope for the given astrological sign."""
    horoscopes = {
        "Aquarius": "Next Tuesday you will befriend a baby otter.",
        "Pisces": "A surprising email will change your afternoon plans.",
        "Aries": "Your boldness will pay off in an unexpected meeting.",
    }
    return f"{sign}: {horoscopes.get(sign, 'The stars are aligning in your favor.')}"

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
openai_client = project.get_openai_client()

# Create a conversation
conversation = openai_client.conversations.create()

# Define the function tool schema
func_tool = FunctionTool(
    name="get_horoscope",
    parameters={
        "type": "object",
        "properties": {
            "sign": {
                "type": "string",
                "description": "An astrological sign like Taurus or Aquarius",
            },
        },
        "required": ["sign"],
        "additionalProperties": False,
    },
    description="Get today's horoscope for an astrological sign.",
    strict=True,
)

# Create agent with function tool
agent = project.agents.create_version(
    agent_name="horoscope-agent",
    definition=PromptAgentDefinition(
        model="gpt-4.1-mini",
        instructions="You are a helpful assistant that provides horoscopes.",
        tools=[func_tool],
    ),
)
print(f"Agent created: {agent.name}")

# Step 1: Send prompt — model will request a function call
response = openai_client.responses.create(
    input="What is my horoscope? I am an Aquarius.",
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

# Step 2: Process function calls from the response
input_list = []
for item in response.output:
    if item.type == "function_call":
        print(f"Model requested function: {item.name}({item.arguments})")
        if item.name == "get_horoscope":
            # Execute the function locally
            result = get_horoscope(**json.loads(item.arguments))
            # Package the result
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({"horoscope": result}),
            })

# Step 3: Submit function results back to the agent
if input_list:
    final_response = openai_client.responses.create(
        input=input_list,
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
    print(f"\nAgent response: {final_response.output_text}")

# Cleanup
project.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
openai_client.conversations.delete(conversation_id=conversation.id)
print("\nCleaned up agent and conversation.")

---
# PART 13: TOOL AUTHENTICATION PATTERNS
---

## 13.1 Authentication by Tool Type

| Tool Type | Auth Method | Details |
|-----------|-----------|--------|
| **Built-in** (Code Interpreter, File Search) | Automatic | No config needed |
| **Azure AI Search** | Project connection | Key-based or keyless (Entra ID) |
| **MCP Servers** | Multiple options | Key-based, Entra (managed identity), OAuth (user passthrough) |
| **OpenAPI** | Anonymous, API key, or managed identity | Defined in tool spec |

### MCP Authentication Example (Key-based)

```python
from azure.ai.projects.models import MCPTool

tool = MCPTool(
    server_label="github",
    server_url="https://api.githubcopilot.com/mcp",
    require_approval="always",
    project_connection_id="my-github-connection",  # stores the credential
)
```

### OpenAPI Authentication Example (Anonymous)

```python
from azure.ai.projects.models import (
    OpenApiTool,
    OpenApiFunctionDefinition,
    OpenApiAnonymousAuthDetails,
)

weather_tool = OpenApiTool(
    openapi=OpenApiFunctionDefinition(
        name="get_weather",
        spec=openapi_spec,
        description="Retrieve weather information.",
        auth=OpenApiAnonymousAuthDetails(),
    )
)
```

> **Best Practice:** Start with Entra authentication if the server supports it — eliminates secret management.

---
# PART 14: SECURITY BEST PRACTICES SUMMARY
---

| Practice | Details |
|----------|--------|
| **Least privilege** | Assign only permissions the agent needs; prefer narrow scopes |
| **No embedded secrets** | Use managed identities and connections; store secrets in Key Vault |
| **Identity awareness** | Shared project identity in dev; distinct identity after publishing |
| **RBAC reassignment** | Always reassign roles after publishing — they don't transfer |
| **Tool output validation** | Treat tool outputs as untrusted input |
| **Audit non-Microsoft tools** | Review data handling for external MCP servers |
| **No secrets in prompts** | Never include keys, tokens, or credentials in prompts |
| **No secrets in logs** | Avoid logging secrets in traces or application logs |
| **API key auth not supported** | Use Entra ID for agent endpoints |

---
# PART 15: INTERVIEW Q&A — KEY CONCEPTS
---

### Q1: What are the three types of agents in Microsoft Foundry?
**A:** (1) **Prompt-based** — declarative, single agent with model + instructions + tools, editable in portal. (2) **Workflow** — visual orchestrator for multi-agent coordination with branching logic and human-in-loop. (3) **Hosted** (preview) — containerized custom code with any framework, deployed as container images with managed infrastructure.

---

### Q2: What happens to an agent's identity when you publish it?
**A:** During development, all unpublished agents in a project share a **single project identity**. When published, the agent gets its own **distinct Entra agent identity** and **blueprint**. Critical: **permissions do NOT transfer** — you must reassign RBAC roles to the new identity for any downstream resources.

---

### Q3: Explain the 4-stage OAuth 2.0 token exchange for agent tool calls.
**A:** (1) **Blueprint Authentication** — Agent Service authenticates the blueprint to Entra ID. (2) **Agent Identity Token** — Entra ID issues a token for the specific agent identity. (3) **Scoped Token Request** — Agent Service exchanges the agent token for one scoped to the downstream service's **audience** (e.g., `https://storage.azure.com`). (4) **Authenticated Tool Call** — The scoped token is passed to the MCP/A2A endpoint, which validates it against RBAC roles.

---

### Q4: What is the difference between the Responses protocol and Invocations protocol?
**A:** **Responses** is OpenAI-compatible, platform-manages conversation history, streaming, and background execution — best for chatbots and Q&A. **Invocations** accepts arbitrary JSON payloads, gives full HTTP/SSE control — best for webhooks (GitHub, Stripe), non-conversational processing, and custom streaming protocols (AG-UI).

---

### Q5: What is a Toolbox and why would you use one?
**A:** A Toolbox bundles multiple tools (MCP servers, Web Search, AI Search, Code Interpreter, etc.) into a **single MCP-compatible endpoint**. Benefits: (1) Configure tools once, share across agents. (2) Centralized auth — credential injection and token refresh managed by platform. (3) Versioning — test new versions, promote to default without code changes. (4) Framework-agnostic — any MCP-compatible runtime can consume it.

---

### Q6: How do structured inputs work?
**A:** Structured inputs use handlebar syntax (`{{variableName}}`) in agent definitions. You define input schemas under `structured_inputs` with name, description, type, and optional defaults. At runtime, supply values that replace placeholders — customize instructions, vector store IDs, file IDs, MCP URLs, and AI Search filters **without creating new agent versions**.

---

### Q7: What is the difference between RAG and Agentic RAG?
**A:** Traditional RAG uses a single query to retrieve from an index. **Agentic RAG** uses a model to break complex inputs into multiple focused subqueries, runs them in parallel, and returns structured grounding data with citations. It supports context-aware query planning (multi-turn aware), semantic ranking, and optional answer synthesis.

---

### Q8: What are the key defaults for vector stores in file search?
**A:** Chunk size: 800 tokens, chunk overlap: 400 tokens, embedding model: text-embedding-3-large at 256 dimensions, max chunks in context: 20, max files per vector store: 10,000, max 1 vector store per agent and 1 per conversation. Conversation vector stores expire after 7 days of inactivity.

---

### Q9: How does function calling differ from MCP tools?
**A:** **Function calling**: You define function schemas, the model outputs a function call request, YOUR app executes the function locally and returns the result. The app is the executor. **MCP tools**: The model calls a tool hosted on a remote MCP server endpoint — the server executes the tool. The server is the executor. Function calling is best for custom logic; MCP is best for shared tools maintained by another team.

---

### Q10: What is the `tool_choice` parameter and when would you use each value?
**A:** `auto` — model decides (default, good for general use). `required` — model MUST call at least one tool (use for deterministic behavior when you know a tool should be called). `none` — model cannot call tools (use when you want text-only output).

---

### Q11: What are the security concerns specific to agent identity?
**A:** (1) Agent identities are distinct from human identities — helps audit AI vs. human operations. (2) Federated credentials (managed identity) are recommended over client secrets — Azure handles rotation. (3) Shared project identity has broader blast radius — publish agents that need tighter controls. (4) Always use least-privilege RBAC with narrow scopes. (5) An incorrect audience value causes auth failures even with correct RBAC.

---

### Q12: What happens to sessions and state in Hosted agents?
**A:** Each session gets a VM-isolated sandbox with persistent filesystem (`$HOME` and `/files`). Sessions have a 15-minute idle timeout, after which compute is deprovisioned but state is persisted. When the session is resumed, new compute is provisioned and state is restored. Sessions persist for up to 30 days. For the Responses protocol, conversation ID is primary (platform manages history). For Invocations protocol, session ID is primary (you manage state).

---

### Q13: How do you set up a private tool catalog?
**A:** (1) Create an **Azure API Center** resource. (2) Register your remote MCP servers in API Center. (3) Optionally configure authentication (API Key, OAuth, HTTP bearer). (4) Assign **Azure API Center Data Reader** role to developers. (5) Developers discover tools in Foundry Portal at Build > Tools. The API Center name becomes the catalog name.

---

### Q14: What is the publishing model for Agent Applications?
**A:** An **Agent Application** wraps an agent version with: a stable endpoint URL, distinct Entra identity, authorization policy (RBAC/Bot Service), and traffic routing. A **Deployment** is a child resource referencing a specific agent version. Currently supports one active deployment with 100% traffic. **Publisher-pays model** — the Foundry project owner pays based on deployed infrastructure, not per-call consumption.

---

### Q15: What are workflows' Power Fx variables and how are they scoped?
**A:** Power Fx uses Excel-like formulas. Variables need scope prefixes: `System.` for system variables (e.g., `System.User.Language`, `System.LastMessage.Text`, `System.Conversation.Id`) and `Local.` for local variables. Common formulas include `Upper()`, `Text()`, `If()`, `ParseJSON()`, `ForAll()`. Expressions can be used in if/else conditions and variable transformations.

---
## Clean Up Resources

In [ ]:
# Clean up agents created in this notebook
try:
    project.agents.delete_version(agent_name="lifecycle-demo-agent", agent_version=agent.version)
    print("Deleted lifecycle-demo-agent")
except Exception as e:
    print(f"Cleanup note: {e}")

try:
    openai_client.conversations.delete(conversation_id=conversation.id)
    print("Deleted conversation")
except Exception as e:
    print(f"Cleanup note: {e}")

---
## Summary

In this notebook you learned:

- [x] The 8-step agent development lifecycle (create, test, tools, version, trace, evaluate, publish, monitor)
- [x] Three agent types: Prompt-based, Workflow, Hosted
- [x] Runtime components: Agents (named/versioned), Conversations (history), Responses (output)
- [x] Agent identity: Entra ID service principals, 4-stage OAuth token exchange, shared vs. distinct identity
- [x] Agent Applications: Publishing, stable endpoints, protocols (Responses/Activity/Invocations/A2A)
- [x] Configure agents: Version routing, protocols, authorization schemes, agent cards
- [x] RAG patterns: Traditional vs. Agentic RAG, indexes, embeddings
- [x] Vector stores: Chunking, ingestion, readiness, expiration policies
- [x] Tool catalog: 12+ built-in tools, 4 custom tool types
- [x] Tool best practices: `tool_choice`, secure usage, troubleshooting
- [x] Private tool catalog via Azure API Center
- [x] Toolbox: Centralized tool bundles with single MCP endpoint
- [x] Structured inputs: Runtime customization with handlebar templates
- [x] Azure AI Search tool with streaming citations
- [x] Code Interpreter: Sandboxed Python for data analysis and chart generation
- [x] Function calling: Custom functions with local execution
- [x] Security best practices and identity management
- [x] 15 interview-ready Q&A covering all core concepts

### Useful Links

- [Agent Development Lifecycle](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/development-lifecycle)
- [Agent Identity Concepts](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/agent-identity)
- [Hosted Agents](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents)
- [Build a Workflow](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/workflow)
- [Agent Applications](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/agent-applications)
- [RAG Concepts](https://learn.microsoft.com/en-us/azure/foundry/concepts/retrieval-augmented-generation)
- [Tool Catalog](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/tool-catalog)
- [Tool Best Practices](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/tool-best-practice)
- [Toolbox](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/toolbox)
- [Structured Inputs](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/structured-inputs)
- [Azure AI Search Tool](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/ai-search)
- [Code Interpreter](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/code-interpreter)
- [Function Calling](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling)